In [111]:
# Copyright 2020 , Gurobi Optimization , LLC
# Solve a multi - commodity flow problem . Two products ('Methane'and'Hydrogen')
# are produced in 2 countries ('Denmark'and'Poland') and must be sent to
# consumption in 3 countries ('Germany','Norway', and'Netherlands') to
# satisfy demand ('inflow [h , i ]').
#
# Flows on the transportation network must respect arc capacity constraints
# ('capacity [i , j ]'). The objective is to minimize the sum of the arc
# transportation costs ('cost [i , j ]').
import gurobipy as gp
from gurobipy import GRB
# Base data
commodities = ['Methane' ,'Hydrogen']
nodes = ['Denmark','Poland','Germany','Norway','Netherlands']
arc_capacity_data = {
    ('Denmark', 'Germany', 'Methane'): 100,
    ('Denmark', 'Germany', 'Hydrogen'): 150,
    ('Denmark', 'Norway', 'Methane'): 80,
    ('Denmark', 'Norway', 'Hydrogen'): 120,
    ('Denmark', 'Netherlands', 'Methane'): 120,
    ('Denmark', 'Netherlands', 'Hydrogen'): 180,
    ('Poland', 'Germany', 'Methane'): 120,
    ('Poland', 'Germany', 'Hydrogen'): 180,
    ('Poland', 'Norway', 'Methane'): 120,
    ('Poland', 'Norway', 'Hydrogen'): 150,
    ('Poland', 'Netherlands', 'Methane'): 120,
    ('Poland', 'Netherlands', 'Hydrogen'): 160
}

# Reverse arcs
reverse_arcs = {(j, i, k): cap for (i, j, k), cap in arc_capacity_data.items()}

arcs, capacity = gp.multidict({**arc_capacity_data, **reverse_arcs})
                             
    # Cost for triplets commodity - source - destination
cost = {
    ('Methane','Denmark','Germany'): 10 ,
#    ('Methane','Germany','Denmark'): 10 , #added for enable flow in other direction
    ('Methane','Denmark','Norway'): 20 ,
    ('Methane','Denmark','Netherlands'): 60 ,
    ('Methane','Poland','Germany'): 40 ,
    ('Methane','Poland','Norway'): 40 ,
    ('Methane','Poland','Netherlands'): 30 ,
    ('Hydrogen','Denmark','Germany'): 20 ,
    ('Hydrogen','Denmark','Norway'): 20 ,
    ('Hydrogen','Denmark','Netherlands'): 80 ,
    ('Hydrogen','Poland','Germany'): 60 ,
    ('Hydrogen','Poland','Norway'): 70 ,
    ('Hydrogen','Poland','Netherlands'): 30}
# Demand for pairs of commodity - city
inflow = {
    ('Methane','Denmark'): -50 ,
    ('Methane','Poland'): 0 ,
    ('Methane','Germany'): 50 ,
    ('Methane','Norway'): 0 ,
    ('Methane','Netherlands'): 0 ,
    ('Hydrogen','Denmark'): 60 ,
    ('Hydrogen','Poland'): 40 ,
    ('Hydrogen','Germany'): -40 ,
    ('Hydrogen','Norway'): -30 ,
    ('Hydrogen','Netherlands'): -30}
# Create optimization model
m = gp . Model ('netflow')
# Create variables
flow = m . addVars ( commodities , arcs , obj = cost , name = " flow " )

# Arc-capacity constraints with commodity-specific capacities
m.addConstrs(
    (flow.sum('*', i, j) <= capacity[i, j, h] for i, j, h in arcs), "cap"
)

# Flow - conservation constraints
m . addConstrs (
    ( flow . sum (h ,'*', j ) + inflow [h , j ] == flow . sum (h , j ,'*') for h in commodities for j in nodes ) , " node " )

# Compute optimal solution
m . optimize ()
# Print solution
if m.status == GRB.OPTIMAL:
    solution = m.getAttr('x', flow)
    for h in commodities:
        print('\nOptimal flows for %s:' % h)
        for i, j, k in arcs:
            if solution[h, i, j, k] > 0:
                print('%s -> %s (Commodity: %s): %g' % (i, j, k, solution[h, i, j, k]))

Gurobi Optimizer version 11.0.0 build v11.0.0rc2 (win64 - Windows 11.0 (22621.2))

CPU model: Intel(R) Core(TM) i5-8265U CPU @ 1.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 34 rows, 48 columns and 192 nonzeros
Model fingerprint: 0x12a632d6
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [0e+00, 0e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [3e+01, 2e+02]
Presolve removed 16 rows and 28 columns
Presolve time: 0.02s
Presolved: 18 rows, 20 columns, 52 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   6.250000e+00   0.000000e+00      0s
       1    0.0000000e+00   0.000000e+00   0.000000e+00      0s

Solved in 1 iterations and 0.04 seconds (0.00 work units)
Optimal objective  0.000000000e+00

Optimal flows for Methane:
Germany -> Denmark (Commodity: Methane): 50

Optimal flows for Hydrogen:
Denmark -> No

In [70]:
import pandas as pd

# Print solution
if m.status == GRB.OPTIMAL:
    solution = m.getAttr('x', flow)
    
    # Create a list to store the data
    data = []

    for h in commodities:
        for i, j, k in arcs:
            if solution[h, i, j, k] > 0:
                data.append({
                    'Commodity': h,
                    'Source': i,
                    'Target': j,
                    'Value': solution[h, i, j, k]
                })

    # Create a DataFrame from the collected data
    solution_df = pd.DataFrame(data)

# Print the DataFrame
solution_df

,Commodity,Source,Target,Value
0,Methane,Germany,Denmark,50.0
1,Hydrogen,Denmark,Norway,30.0
2,Hydrogen,Denmark,Netherlands,30.0
3,Hydrogen,Poland,Germany,40.0


In [60]:
import gurobipy as gp
from gurobipy import GRB

# Base data
commodities = ['Methane', 'Hydrogen']
nodes = ['Denmark', 'Poland', 'Germany', 'Norway', 'Netherlands']
arc_capacity_data = {
    ('Denmark', 'Germany', 'Methane'): 100,
    ('Denmark', 'Germany', 'Hydrogen'): 150,
    ('Denmark', 'Norway', 'Methane'): 80,
    ('Denmark', 'Norway', 'Hydrogen'): 120,
    ('Denmark', 'Netherlands', 'Methane'): 120,
    ('Denmark', 'Netherlands', 'Hydrogen'): 180,
    ('Poland', 'Germany', 'Methane'): 120,
    ('Poland', 'Germany', 'Hydrogen'): 180,
    ('Poland', 'Norway', 'Methane'): 120,
    ('Poland', 'Norway', 'Hydrogen'): 150,
    ('Poland', 'Netherlands', 'Methane'): 120,
    ('Poland', 'Netherlands', 'Hydrogen'): 160
}

# Reverse arcs
reverse_arcs = {(j, i, k): cap for (i, j, k), cap in arc_capacity_data.items()}

arcs, capacity = gp.multidict({**arc_capacity_data, **reverse_arcs})

# Cost for triplets commodity - source - destination
cost = {
    ('Methane', 'Denmark', 'Germany'): 10,
    ('Methane', 'Germany', 'Denmark'): 10,  # Enable flow in both directions
    ('Methane', 'Denmark', 'Norway'): 20,
    ('Methane', 'Denmark', 'Netherlands'): 60,
    ('Methane', 'Poland', 'Germany'): 40,
    ('Methane', 'Poland', 'Norway'): 40,
    ('Methane', 'Poland', 'Netherlands'): 30,
    ('Hydrogen', 'Denmark', 'Germany'): 20,
    ('Hydrogen', 'Germany', 'Denmark'): 20,  # Enable flow in both directions
    ('Hydrogen', 'Denmark', 'Norway'): 20,
    ('Hydrogen', 'Denmark', 'Netherlands'): 80,
    ('Hydrogen', 'Poland', 'Germany'): 60,
    ('Hydrogen', 'Poland', 'Norway'): 70,
    ('Hydrogen', 'Poland', 'Netherlands'): 30
}

# Demand for pairs of commodity - city
inflow = {
    ('Methane', 'Denmark'): -50,
    ('Methane', 'Poland'): 0,
    ('Methane', 'Germany'): 50,
    ('Methane', 'Norway'): 0,
    ('Methane', 'Netherlands'): 0,
    ('Hydrogen', 'Denmark'): 60,
    ('Hydrogen', 'Poland'): 40,
    ('Hydrogen', 'Germany'): -40,
    ('Hydrogen', 'Norway'): -30,
    ('Hydrogen', 'Netherlands'): -30
}

# Create optimization model
m = gp.Model('netflow')

# Create variables
flow = m.addVars(commodities, arcs, obj=cost, name="flow")

# Arc-capacity constraints with commodity-specific capacities
m.addConstrs(
    (flow.sum('*', i, j, k) <= capacity[i, j, k] for i, j, k in arcs), "cap"
)

# Flow - conservation constraints
m.addConstrs(
    (flow.sum(h, '*', j, k) + inflow[h, j] == flow.sum(h, j, '*', k) for h in commodities for j in nodes for k in nodes),
    "node"
)

# Compute optimal solution
m.optimize()

# Print solution
if m.status == GRB.OPTIMAL:
    solution = m.getAttr('x', flow)
    for h in commodities:
        print('\nOptimal flows for %s:' % h)
        for i, j, k in arcs:
            if solution[h, i, j, k] > 0:
                print('%s -> %s (Commodity: %s): %g' % (i, j, k, solution[h, i, j, k]))

Gurobi Optimizer version 11.0.0 build v11.0.0rc2 (win64 - Windows 11.0 (22621.2))

CPU model: Intel(R) Core(TM) i5-8265U CPU @ 1.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 74 rows, 48 columns and 48 nonzeros
Model fingerprint: 0xe934c21b
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [0e+00, 0e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [3e+01, 2e+02]
Presolve time: 0.01s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Infeasible model


In [55]:
reverse_arcs

{('Germany', 'Denmark', 'Methane'): 100,
 ('Germany', 'Denmark', 'Hydrogen'): 150,
 ('Norway', 'Denmark', 'Methane'): 80,
 ('Norway', 'Denmark', 'Hydrogen'): 120,
 ('Netherlands', 'Denmark', 'Methane'): 120,
 ('Netherlands', 'Denmark', 'Hydrogen'): 180,
 ('Germany', 'Poland', 'Methane'): 120,
 ('Germany', 'Poland', 'Hydrogen'): 180,
 ('Norway', 'Poland', 'Methane'): 120,
 ('Norway', 'Poland', 'Hydrogen'): 150,
 ('Netherlands', 'Poland', 'Methane'): 120,
 ('Netherlands', 'Poland', 'Hydrogen'): 160}

In [83]:
# Copyright 2023, Gurobi Optimization, LLC

# Solve a multi-commodity flow problem.  Two products ('Pencils' and 'Pens')
# are produced in 2 cities ('Detroit' and 'Denver') and must be sent to
# warehouses in 3 cities ('Boston', 'New York', and 'Seattle') to
# satisfy supply/demand ('inflow[h,i]').
#
# Flows on the transportation network must respect arc capacity constraints
# ('capacity[i,j]'). The objective is to minimize the sum of the arc
# transportation costs ('cost[i,j]').

import gurobipy as gp
from gurobipy import GRB

# Base data
commodities = ["Pencils", "Pens"]
nodes = ["Detroit", "Denver", "Boston", "New York", "Seattle"]

arcs, capacity = gp.multidict(
    {
        ("Detroit", "Boston"): 100,
        ("Detroit", "New York"): 80,
        ("Detroit", "Seattle"): 120,
        ("Denver", "Boston"): 120,
        ("Denver", "New York"): 120,
        ("Denver", "Seattle"): 120,
    }
)

# Cost for triplets commodity-source-destination
cost = {
    ("Pencils", "Detroit", "Boston"): 10,
    ("Pencils", "Detroit", "New York"): 20,
    ("Pencils", "Detroit", "Seattle"): 60,
    ("Pencils", "Denver", "Boston"): 40,
    ("Pencils", "Denver", "New York"): 40,
    ("Pencils", "Denver", "Seattle"): 30,
    ("Pens", "Detroit", "Boston"): 20,
    ("Pens", "Detroit", "New York"): 20,
    ("Pens", "Detroit", "Seattle"): 80,
    ("Pens", "Denver", "Boston"): 60,
    ("Pens", "Denver", "New York"): 70,
    ("Pens", "Denver", "Seattle"): 30,
}

# Supply (> 0) and demand (< 0) for pairs of commodity-city
inflow = {
    ("Pencils", "Detroit"): 50,
    ("Pencils", "Denver"): 60,
    ("Pencils", "Boston"): -50,
    ("Pencils", "New York"): -50,
    ("Pencils", "Seattle"): -10,
    ("Pens", "Detroit"): 60,
    ("Pens", "Denver"): 40,
    ("Pens", "Boston"): -40,
    ("Pens", "New York"): -30,
    ("Pens", "Seattle"): -30,
}

# Create optimization model
m = gp.Model("netflow")

# Create variables
flow = m.addVars(commodities, arcs, obj=cost, name="flow")

m.setObjective(gp.quicksum(flow[h, i, j] * cost[h, i, j] for h, i, j in flow), GRB.MINIMIZE)

# Arc-capacity constraints
m.addConstrs((flow.sum("*", i, j) <= capacity[i, j] for i, j in arcs), "cap")

# Equivalent version using Python looping
# for i, j in arcs:
#   m.addConstr(sum(flow[h, i, j] for h in commodities) <= capacity[i, j],
#               "cap[%s, %s]" % (i, j))


# Flow-conservation constraints
m.addConstrs(
    (
        flow.sum(h, "*", j) + inflow[h, j] == flow.sum(h, j, "*")
        for h in commodities
        for j in nodes
    ),
    "node",
)

# Alternate version:
# m.addConstrs(
#   (gp.quicksum(flow[h, i, j] for i, j in arcs.select('*', j)) + inflow[h, j] ==
#     gp.quicksum(flow[h, j, k] for j, k in arcs.select(j, '*'))
#     for h in commodities for j in nodes), "node")

# Compute optimal solution
m.optimize()

# Print solution
if m.Status == GRB.OPTIMAL:
    solution = m.getAttr("X", flow)
    for h in commodities:
        print(f"\nOptimal flows for {h}:")
        for i, j in arcs:
            if solution[h, i, j] > 0:
                print(f"{i} -> {j}: {solution[h, i, j]:g}")

Gurobi Optimizer version 11.0.0 build v11.0.0rc2 (win64 - Windows 11.0 (22621.2))

CPU model: Intel(R) Core(TM) i5-8265U CPU @ 1.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 16 rows, 12 columns and 36 nonzeros
Model fingerprint: 0xc43e5943
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+01, 8e+01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+01, 1e+02]
Presolve removed 16 rows and 12 columns
Presolve time: 0.01s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    5.5000000e+03   0.000000e+00   2.000000e+01      0s
Extra simplex iterations after uncrush: 1
       1    5.5000000e+03   0.000000e+00   0.000000e+00      0s

Solved in 1 iterations and 0.02 seconds (0.00 work units)
Optimal objective  5.500000000e+03

Optimal flows for Pencils:
Detroit -> Boston: 50
Denver -> New York: 50
Denver

In [21]:
import gurobipy as gp
from gurobipy import GRB

# Base data
commodities = ["Pencils", "Pens"]
nodes = ["Detroit", "Denver", "Boston", "New York", "Seattle"]

# Format: (commodity, source, destination): capacity
start_cap = {
    ("Pencils", "Detroit", "Boston"): 100,
    ("Pencils", "Detroit", "New York"): 80,
    ("Pencils", "Detroit", "Seattle"): 120,
    ("Pencils", "Denver", "Boston"): 120,
    ("Pencils", "Denver", "New York"): 120,
    ("Pencils", "Denver", "Seattle"): 120,
    ("Pens", "Detroit", "Boston"): 200,  # Example different capacity for Pens
    ("Pens", "Detroit", "New York"): 150,
    ("Pens", "Detroit", "Seattle"): 180,
    ("Pens", "Denver", "Boston"): 160,
    ("Pens", "Denver", "New York"): 120,
    ("Pens", "Denver", "Seattle"): 100,
}

# Reverse arcs
reverse_cap = {(k, j, i): cap for (k, i, j), cap in start_cap.items()}

arcs, capacity = gp.multidict({**start_cap, **reverse_cap})


# Cost for triplets commodity-source-destination
cost_start = {
    ("Pencils", "Detroit", "Boston"): 10,
    ("Pencils", "Detroit", "New York"): 20,
    ("Pencils", "Detroit", "Seattle"): 60,
    ("Pencils", "Denver", "Boston"): 40,
    ("Pencils", "Denver", "New York"): 40,
    ("Pencils", "Denver", "Seattle"): 30,
    ("Pens", "Detroit", "Boston"): 20,
    ("Pens", "Detroit", "New York"): 20,
    ("Pens", "Detroit", "Seattle"): 80,
    ("Pens", "Denver", "Boston"): 60,
    ("Pens", "Denver", "New York"): 70,
    ("Pens", "Denver", "Seattle"): 30,
}

reverse_cost = {(k, j, i): cost for (k, i, j), cost in cost_start.items()}

cost = {**cost_start, **reverse_cost}

# Supply (> 0) and demand (< 0) for pairs of commodity-city
inflow = {
    ("Pencils", "Detroit"): 50,
    ("Pencils", "Denver"): 0,
    ("Pencils", "Boston"): -50,
    ("Pencils", "New York"): 0,
    ("Pencils", "Seattle"): 0,
    ("Pens", "Detroit"): 60,
    ("Pens", "Denver"): 40,
    ("Pens", "Boston"): -40,
    ("Pens", "New York"): -30,
    ("Pens", "Seattle"): -30,
}

# Create optimization model
m = gp.Model("netflow")

# Create variables
flow = m.addVars(commodities, arcs, name="flow")

m.setObjective(gp.quicksum(flow[h, i, j] * cost[h, i, j] for h in commodities for i, j in arcs), GRB.MINIMIZE)

# Arc-capacity constraints
m.addConstrs((flow.sum(h, i, j) <= capacity[h, i, j] for h, i, j in flow), "cap")

# Flow-conservation constraints
m.addConstrs(
    (
        flow.sum(h, "*", j) + inflow[h, j] == flow.sum(h, j, "*")
        for h in commodities
        for j in nodes
    ),
    "node",
)

# Compute optimal solution
m.optimize()

# Print solution
if m.Status == GRB.OPTIMAL:
    solution = m.getAttr("X", flow)
    for h in commodities:
        print(f"\nOptimal flows for {h}:")
        for i, j in arcs:
            if solution[h, i, j] > 0:
                print(f"{i} -> {j}: {solution[h, i, j]:g}")

ValueError: too many values to unpack (expected 2)

In [19]:
capacity

{('Pencils', 'Detroit', 'Boston'): 100,
 ('Pencils', 'Detroit', 'New York'): 80,
 ('Pencils', 'Detroit', 'Seattle'): 120,
 ('Pencils', 'Denver', 'Boston'): 120,
 ('Pencils', 'Denver', 'New York'): 120,
 ('Pencils', 'Denver', 'Seattle'): 120,
 ('Pens', 'Detroit', 'Boston'): 200,
 ('Pens', 'Detroit', 'New York'): 150,
 ('Pens', 'Detroit', 'Seattle'): 180,
 ('Pens', 'Denver', 'Boston'): 160,
 ('Pens', 'Denver', 'New York'): 120,
 ('Pens', 'Denver', 'Seattle'): 100,
 ('Pencils', 'Boston', 'Detroit'): 100,
 ('Pencils', 'New York', 'Detroit'): 80,
 ('Pencils', 'Seattle', 'Detroit'): 120,
 ('Pencils', 'Boston', 'Denver'): 120,
 ('Pencils', 'New York', 'Denver'): 120,
 ('Pencils', 'Seattle', 'Denver'): 120,
 ('Pens', 'Boston', 'Detroit'): 200,
 ('Pens', 'New York', 'Detroit'): 150,
 ('Pens', 'Seattle', 'Detroit'): 180,
 ('Pens', 'Boston', 'Denver'): 160,
 ('Pens', 'New York', 'Denver'): 120,
 ('Pens', 'Seattle', 'Denver'): 100}

In [106]:
start_cap

{('Pencils', 'Detroit', 'Boston'): 100,
 ('Pencils', 'Detroit', 'New York'): 80,
 ('Pencils', 'Detroit', 'Seattle'): 120,
 ('Pencils', 'Denver', 'Boston'): 120,
 ('Pencils', 'Denver', 'New York'): 120,
 ('Pencils', 'Denver', 'Seattle'): 120,
 ('Pens', 'Detroit', 'Boston'): 200,
 ('Pens', 'Detroit', 'New York'): 150,
 ('Pens', 'Detroit', 'Seattle'): 180,
 ('Pens', 'Denver', 'Boston'): 160,
 ('Pens', 'Denver', 'New York'): 120,
 ('Pens', 'Denver', 'Seattle'): 100}

In [11]:
import gurobipy as gp
from gurobipy import GRB

# Base data
commodities = ["Pencils", "Pens"]
nodes = ["Detroit", "Denver", "Boston", "New York", "Seattle"]

arcs, capacity = gp.multidict(
    {
        ("Detroit", "Boston"): 100,
        ("Detroit", "New York"): 80,
        ("Detroit", "Seattle"): 120,
        ("Denver", "Boston"): 120,
        ("Denver", "New York"): 120,
        ("Denver", "Seattle"): 120,
    }
)

# Cost for triplets commodity-source-destination
cost = {
    ("Pencils", "Detroit", "Boston"): 10,
    ("Pencils", "Detroit", "New York"): 20,
    ("Pencils", "Detroit", "Seattle"): 60,
    ("Pencils", "Denver", "Boston"): 40,
    ("Pencils", "Denver", "New York"): 40,
    ("Pencils", "Denver", "Seattle"): 30,
    ("Pens", "Detroit", "Boston"): 20,
    ("Pens", "Detroit", "New York"): 20,
    ("Pens", "Detroit", "Seattle"): 80,
    ("Pens", "Denver", "Boston"): 60,
    ("Pens", "Denver", "New York"): 70,
    ("Pens", "Denver", "Seattle"): 30,
}

# Supply (> 0) and demand (< 0) for pairs of commodity-city
inflow = {
    ("Pencils", "Detroit"): 50,
    ("Pencils", "Denver"): 60,
    ("Pencils", "Boston"): -50,
    ("Pencils", "New York"): -50,
    ("Pencils", "Seattle"): -10,
    ("Pens", "Detroit"): 60,
    ("Pens", "Denver"): 40,
    ("Pens", "Boston"): -40,
    ("Pens", "New York"): -30,
    ("Pens", "Seattle"): -30,
}

# Create optimization model
m = gp.Model("netflow")

# Create variables
flow = m.addVars(commodities, arcs, name="flow")

m.setObjective(gp.quicksum(flow[h, i, j] * cost[h, i, j] for h in commodities for i, j in arcs), GRB.MINIMIZE)

# Arc-capacity constraints
m.addConstrs((flow.sum("*", i, j) <= capacity[i, j] for i, j in arcs), "cap")

# Flow-conservation constraints
m.addConstrs(
    (
        flow.sum(h, "*", j) + inflow[h, j] == flow.sum(h, j, "*")
        for h in commodities
        for j in nodes
    ),
    "node",
)

# Compute optimal solution
m.optimize()

# Print solution
if m.Status == GRB.OPTIMAL:
    solution = m.getAttr("X", flow)
    for h in commodities:
        print(f"\nOptimal flows for {h}:")
        for i, j in arcs:
            if solution[h, i, j] > 0:
                print(f"{i} -> {j}: {solution[h, i, j]:g}")

Gurobi Optimizer version 11.0.0 build v11.0.0rc2 (win64 - Windows 11.0 (22621.2))

CPU model: Intel(R) Core(TM) i5-8265U CPU @ 1.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 16 rows, 12 columns and 36 nonzeros
Model fingerprint: 0xc43e5943
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+01, 8e+01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+01, 1e+02]
Presolve removed 16 rows and 12 columns
Presolve time: 0.02s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    5.5000000e+03   0.000000e+00   2.000000e+01      0s
Extra simplex iterations after uncrush: 1
       1    5.5000000e+03   0.000000e+00   0.000000e+00      0s

Solved in 1 iterations and 0.03 seconds (0.00 work units)
Optimal objective  5.500000000e+03

Optimal flows for Pencils:
Detroit -> Boston: 50
Denver -> New York: 50
Denver

In [7]:
import gurobipy as gp
from gurobipy import GRB

# Base data
commodities = ["Pencils", "Pens"]
nodes = ["Detroit", "Denver", "Boston", "New York", "Seattle"]

# Arcs
arcs_base = [
    ("Detroit", "Boston"),
    ("Detroit", "New York"),
    ("Detroit", "Seattle"),
    ("Denver", "Boston"),
    ("Denver", "New York"),
    ("Denver", "Seattle"),
]

# Reverse arcs
arcs_reverse = [(destination, source) for source, destination in arcs_base]

# Combined arcs
arcs = arcs_base + arcs_reverse

# Capacity for quadruples commodity-source-destination
capacity_base = {
    ("Pencils", "Detroit", "Boston"): 100,
    ("Pencils", "Detroit", "New York"): 80,
    ("Pencils", "Detroit", "Seattle"): 120,
    ("Pencils", "Denver", "Boston"): 120,
    ("Pencils", "Denver", "New York"): 120,
    ("Pencils", "Denver", "Seattle"): 120,
    ("Pens", "Detroit", "Boston"): 150,
    ("Pens", "Detroit", "New York"): 120,
    ("Pens", "Detroit", "Seattle"): 100,
    ("Pens", "Denver", "Boston"): 100,
    ("Pens", "Denver", "New York"): 150,
    ("Pens", "Denver", "Seattle"): 80,
}

# Reverse cap
capacity_reverse = {(k, j, i): cap for (k, i, j), cap in capacity_base.items()}

capacity = {**capacity_base, **capacity_reverse}

# Cost for triplets commodity-source-destination
cost_base = {
    ("Pencils", "Detroit", "Boston"): 10,
    ("Pencils", "Detroit", "New York"): 20,
    ("Pencils", "Detroit", "Seattle"): 60,
    ("Pencils", "Denver", "Boston"): 40,
    ("Pencils", "Denver", "New York"): 40,
    ("Pencils", "Denver", "Seattle"): 30,
    ("Pens", "Detroit", "Boston"): 20,
    ("Pens", "Detroit", "New York"): 20,
    ("Pens", "Detroit", "Seattle"): 80,
    ("Pens", "Denver", "Boston"): 60,
    ("Pens", "Denver", "New York"): 70,
    ("Pens", "Denver", "Seattle"): 30,
}

# Reverse cost
cost_reverse = {(k, j, i): cost for (k, i, j), cost in cost_base.items()}

cost = {**cost_base, **cost_reverse}

# Supply (> 0) and demand (< 0) for pairs of commodity-city
inflow = {
    ("Pencils", "Detroit"): -50,
    ("Pencils", "Denver"): 0,
    ("Pencils", "Boston"): 50,
    ("Pencils", "New York"): 0,
    ("Pencils", "Seattle"): 0,
    ("Pens", "Detroit"): 60,
    ("Pens", "Denver"): 40,
    ("Pens", "Boston"): -40,
    ("Pens", "New York"): -30,
    ("Pens", "Seattle"): -30,
}

# Create optimization model
m = gp.Model("netflow")

# Create variables
flow = m.addVars(commodities, arcs, name="flow")

# Set objective function
m.setObjective(gp.quicksum(flow[h, i, j] * cost[h, i, j] for h in commodities for i, j in arcs), GRB.MINIMIZE)

# Arc-capacity constraints
m.addConstrs((flow.sum(h, i, j) <= capacity[h, i, j] for h in commodities for i, j in arcs), "cap")

# Flow-conservation constraints
m.addConstrs(
    (
        flow.sum(h, "*", j) + inflow[h, j] == flow.sum(h, j, "*")
        for h in commodities
        for j in nodes
    ),
    "node",
)

# Compute optimal solution
m.optimize()

# Print solution
if m.Status == GRB.OPTIMAL:
    solution = m.getAttr("X", flow)
    for h in commodities:
        print(f"\nOptimal flows for {h}:")
        for i, j in arcs:
            if solution[h, i, j] > 0:
                print(f"{i} -> {j}: {solution[h, i, j]:g}")


Gurobi Optimizer version 11.0.0 build v11.0.0rc2 (win64 - Windows 11.0 (22621.2))

CPU model: Intel(R) Core(TM) i5-8265U CPU @ 1.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 34 rows, 24 columns and 72 nonzeros
Model fingerprint: 0xabebf29a
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+01, 8e+01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [3e+01, 2e+02]
Presolve removed 28 rows and 4 columns
Presolve time: 0.01s
Presolved: 6 rows, 20 columns, 28 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   2.375000e+01   0.000000e+00      0s
       5    3.2000000e+03   0.000000e+00   0.000000e+00      0s

Solved in 5 iterations and 0.03 seconds (0.00 work units)
Optimal objective  3.200000000e+03

Optimal flows for Pencils:
Boston -> Detroit: 50

Optimal flows for Pens:
Detroit -> Boston: 30
Detroit -> New York: